# AGENTIC AI
## SESSION 2

Assignment questions and answers


## Question 1

Task 1: AI Technique Decision Engine

Build a Python function recommend_technique(update_frequency,
data_in_external_docs, needs_custom_behaviour, latency_budget) accepting
update_frequency (‘high’/‘low’), data_in_external_docs (‘yes’/‘no’),
needs_custom_behaviour (‘yes’/‘no’), and latency_budget (‘low’/‘high’).

Requirements: - Use a decision tree to map inputs to Prompt Engineering,
RAG, or Fine-Tuning, with at least five distinct rule paths. - Return a
dictionary with keys ‘technique’ and ‘justification’; the justification
must be two sentences. - Test at least four food-delivery scenarios
(e.g. real-time menu recommendations, complaint categorisation, order
confirmation messages, and domain-specific FAQ answering) and print each
recommendation.

**Answer:**

```python
def recommend_technique(
    update_frequency,
    data_in_external_docs,
    needs_custom_behaviour,
    latency_budget,
):
    # Rule Path 1: Low latency budget prioritizes native model speed
    if latency_budget == "low" and needs_custom_behaviour == "yes":
        technique = "Fine-Tuning"
        justification = (
            "A strict low latency budget rules out the overhead of external retrieval steps. "
            "Fine-tuning bakes the desired custom behavior directly into model weights for fast inference."
        )

    # Rule Path 2: External dynamic knowledge requires retrieval
    elif data_in_external_docs == "yes" and update_frequency == "high":
        technique = "RAG"
        justification = (
            "Frequently updated external documents cannot be reliably maintained through model retraining alone. "
            "RAG dynamically fetches the most recent context at query time without altering model weights."
        )

    # Rule Path 3: External static/low-frequency knowledge with high latency budget
    elif data_in_external_docs == "yes" and latency_budget == "high":
        technique = "RAG"
        justification = (
            "Grounding responses on external knowledge bases prevents hallucinations on specific facts. "
            "A high latency budget comfortably accommodates the extra time required for document retrieval."
        )

    # Rule Path 4: Specialized behavior/style on internal static data
    elif needs_custom_behaviour == "yes" and data_in_external_docs == "no":
        technique = "Fine-Tuning"
        justification = (
            "Adapting a model to specialized styles or tasks without external documents is ideal for weight adaptation. "
            "Fine-tuning teaches task-specific formats and behaviors permanently."
        )

    # Rule Path 5: Standard behavior, no external docs, low update frequency
    else:
        technique = "Prompt Engineering"
        justification = (
            "Standard tasks lacking external data dependencies do not justify the cost of retrieval or fine-tuning. "
            "Prompt engineering provides sufficient task guidance using in-context instructions directly."
        )

    return {"technique": technique, "justification": justification}


# Test Scenarios
scenarios = [
    {
        "name": "Real-time menu recommendations",
        "params": {
            "update_frequency": "high",
            "data_in_external_docs": "yes",
            "needs_custom_behaviour": "no",
            "latency_budget": "high",
        },
    },
    {
        "name": "Complaint categorisation",
        "params": {
            "update_frequency": "low",
            "data_in_external_docs": "no",
            "needs_custom_behaviour": "yes",
            "latency_budget": "low",
        },
    },
    {
        "name": "Order confirmation messages",
        "params": {
            "update_frequency": "low",
            "data_in_external_docs": "no",
            "needs_custom_behaviour": "no",
            "latency_budget": "low",
        },
    },
    {
        "name": "Domain-specific FAQ answering",
        "params": {
            "update_frequency": "low",
            "data_in_external_docs": "yes",
            "needs_custom_behaviour": "no",
            "latency_budget": "high",
        },
    },
]

for scenario in scenarios:
    result = recommend_technique(**scenario["params"])
    print(f"Scenario: {scenario['name']}")
    print(f"Technique: {result['technique']}")
    print(f"Justification: {result['justification']}\n")
```

## Question 2

Task 2: Tool-Calling Delivery Agent

Build a Python class FoodDeliveryAgent that routes customer queries to
the correct internal tool and executes it.

Requirements: - Define check_restaurant_status(name),
get_estimated_delivery_time(order_id), apply_discount(order_id, reason),
and file_complaint(order_id, issue) outside the class; each returns a
formatted string. - Define an MCP-style schema dictionary for each tool
with keys name, description, and parameters. Print all four schemas once
before test queries. - Implement FoodDeliveryAgent.think(query) to parse
keywords and map queries to the correct tool. - Record each (query,
tool_called, result) tuple in an internal log list. - Test at least five
varied queries covering all four tools, and print outputs and the
session log.

**Answer:**

```python
import json
import re

# Tool Definitions
def check_restaurant_status(name):
    return f"Status for '{name}': Open and currently accepting orders."

def get_estimated_delivery_time(order_id):
    return f"Order #{order_id} estimated delivery time is 25-30 minutes."

def apply_discount(order_id, reason):
    return f"Discount applied to Order #{order_id}. Reason: {reason}."

def file_complaint(order_id, issue):
    return f"Complaint filed for Order #{order_id}. Issue logged: {issue}."

# MCP-Style Schemas
schemas = [
    {
        "name": "check_restaurant_status",
        "description": "Check if a restaurant is currently open and accepting orders.",
        "parameters": {
            "type": "object",
            "properties": {"name": {"type": "string", "description": "Restaurant name"}},
            "required": ["name"]
        }
    },
    {
        "name": "get_estimated_delivery_time",
        "description": "Get estimated delivery time for an order.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string", "description": "Order identifier"}},
            "required": ["order_id"]
        }
    },
    {
        "name": "apply_discount",
        "description": "Apply a discount to an existing order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order identifier"},
                "reason": {"type": "string", "description": "Reason for the discount"}
            },
            "required": ["order_id", "reason"]
        }
    },
    {
        "name": "file_complaint",
        "description": "File a customer complaint regarding an order.",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string", "description": "Order identifier"},
                "issue": {"type": "string", "description": "Complaint details"}
            },
            "required": ["order_id", "issue"]
        }
    }
]

# Agent Class
class FoodDeliveryAgent:
    def __init__(self):
        self.log = []

    def think(self, query):
        q = query.lower()
        order_match = re.search(r'#?(\d+)', query)
        order_id = order_match.group(1) if order_match else "Unknown"

        if "status" in q or "open" in q:
            tool_name = "check_restaurant_status"
            name = query.split("status of")[-1].strip("?. ") if "status of" in q else "Target Restaurant"
            result = check_restaurant_status(name)
        elif "time" in q or "eta" in q or "delivery" in q and "status" not in q:
            tool_name = "get_estimated_delivery_time"
            result = get_estimated_delivery_time(order_id)
        elif "discount" in q or "promo" in q or "coupon" in q:
            tool_name = "apply_discount"
            result = apply_discount(order_id, "Customer request / promotional")
        elif "complaint" in q or "wrong" in q or "cold" in q or "issue" in q:
            tool_name = "file_complaint"
            result = file_complaint(order_id, query)
        else:
            tool_name = "none"
            result = "Query could not be routed to a tool."

        self.log.append((query, tool_name, result))
        return result

# Print Schemas
print("=== Tool Schemas ===")
print(json.dumps(schemas, indent=2))
print()

# Test Queries
agent = FoodDeliveryAgent()
test_queries = [
    "What is the status of Burger Town?",
    "When will my order #1042 arrive?",
    "Please apply a discount to order 1042 for late delivery.",
    "I want to file a complaint for order #3055, the food was cold.",
    "Is Pizza Palace open right now?"
]

print("=== Query Execution ===")
for query in test_queries:
    print(f"Query: {query}")
    print(f"Output: {agent.think(query)}\n")

print("=== Session Log ===")
for entry in agent.log:
    print(entry)
```

## Question 3

Task 3: Memory-Aware Strategy Selector

Build an agent with prompt_template_response(query),
rag_lookup_response(query), and fine_tuned_response(query), each
returning a simulated relevant food-delivery response.

Requirements: - Maintain memory keyed by user_id with interaction_count,
preferred_cuisine, and last_complaint (initially None), updating it
after every interaction. - Implement decide_strategy(user_id, query):
select fine-tuned for complaint queries after 3+ interactions, RAG for
menu/item queries, and prompt template otherwise; return strategy and
response. - Simulate at least five interactions for the same user_id,
show strategy changes, and print memory after each interaction.

**Answer:**

```python
memory = {}

def prompt_template_response(query):
    return f"Standard Response: Thank you for your query regarding '{query}'."

def rag_lookup_response(query):
    return f"RAG Response: Retrieved current menu and item details for '{query}'."

def fine_tuned_response(query):
    return f"Fine-Tuned Response: Specialized support resolving your complaint: '{query}'."

def decide_strategy(user_id, query):
    if user_id not in memory:
        memory[user_id] = {
            "interaction_count": 0,
            "preferred_cuisine": None,
            "last_complaint": None
        }
    
    user_mem = memory[user_id]
    q_lower = query.lower()
    
    # Strategy selection
    is_complaint = any(w in q_lower for w in ["cold", "late", "wrong", "bad", "complaint"])
    is_menu = any(w in q_lower for w in ["menu", "item", "burger", "pizza", "sushi", "price"])
    
    if is_complaint and user_mem["interaction_count"] >= 3:
        strategy = "fine_tuned"
        response = fine_tuned_response(query)
    elif is_menu:
        strategy = "rag_lookup"
        response = rag_lookup_response(query)
    else:
        strategy = "prompt_template"
        response = prompt_template_response(query)
        
    # Memory update
    user_mem["interaction_count"] += 1
    if is_complaint:
        user_mem["last_complaint"] = query
    for cuisine in ["italian", "mexican", "japanese", "indian", "american"]:
        if cuisine in q_lower:
            user_mem["preferred_cuisine"] = cuisine.capitalize()

    return strategy, response

# Simulation
user = "user_101"
queries = [
    "What is on the Italian pizza menu?",
    "My order arrived late and cold.",
    "Do you have a sushi item available?",
    "My burger was completely wrong and cold!",
    "Can you help me track my overall order?"
]

for i, q in enumerate(queries, 1):
    print(f"\n--- Interaction {i} ---")
    print(f"Query: {q}")
    strat, resp = decide_strategy(user, q)
    print(f"Strategy Used: {strat}")
    print(f"Response: {resp}")
    print(f"Memory: {memory[user]}")
```

## Question 4

Task 4: Multi-Agent Order Lifecycle System

Build a console-based system with OrderAgent, DispatchAgent, and
SupportAgent classes, each with process(data). OrderAgent creates an
order summary; DispatchAgent assigns a partner and ETA from a simulated
list; SupportAgent classifies feedback as positive, complaint, or
escalate and selects prompt, RAG, or fine-tuned response technique.

Requirements: - Create Coordinator.run_lifecycle(customer_input,
feedback), passing outputs in order OrderAgent → DispatchAgent →
SupportAgent and collecting each output. - Print a structured report
with order summary, dispatch details, feedback classification,
technique, and simulated processing time. - Run at least two scenarios
(e.g. smooth delivery and complaint) and demonstrate different technique
selections.

**Answer:**

```python
import random
import time


class OrderAgent:

    def process(self, data):
        return {
            "order_id": f"ORD-{random.randint(1000, 9999)}",
            "items": data.get("items", []),
            "total_price": data.get("total", 0.0),
        }


class DispatchAgent:

    def __init__(self):
        self.partners = ["ExpressLogistics", "SwiftDelivery", "HyperCargo"]

    def process(self, data):
        return {
            "order_id": data["order_id"],
            "partner": random.choice(self.partners),
            "eta": f"{random.randint(15, 60)} minutes",
        }


class SupportAgent:

    def process(self, data):
        feedback = data.lower()
        if any(w in feedback for w in ["damaged", "never", "terrible", "bad"]):
            classification = "complaint"
            technique = "RAG"
        elif any(w in feedback for w in ["urgent", "lawyer", "refund"]):
            classification = "escalate"
            technique = "fine-tuned"
        else:
            classification = "positive"
            technique = "prompt"

        return {
            "feedback": data,
            "classification": classification,
            "technique": technique,
        }


class Coordinator:

    def __init__(self):
        self.order_agent = OrderAgent()
        self.dispatch_agent = DispatchAgent()
        self.support_agent = SupportAgent()

    def run_lifecycle(self, customer_input, feedback):
        start_time = time.time()

        order_output = self.order_agent.process(customer_input)
        dispatch_output = self.dispatch_agent.process(order_output)
        support_output = self.support_agent.process(feedback)

        elapsed_time = round(time.time() - start_time + 0.25, 2)

        print("\n--- Order Lifecycle Report ---")
        print(f"Order Summary:       {order_output}")
        print(
            f"Dispatch Details:    Partner: {dispatch_output['partner']}, ETA: {dispatch_output['eta']}"
        )
        print(f"Feedback Category:   {support_output['classification']}")
        print(f"Response Technique:  {support_output['technique']}")
        print(f"Processing Time:     {elapsed_time}s")
        print("------------------------------")


# Scenario Execution
coordinator = Coordinator()

# Scenario 1: Smooth Delivery
print("Running Scenario 1: Smooth Delivery")
coordinator.run_lifecycle(
    customer_input={"items": ["Keyboard", "Mouse"], "total": 45.99},
    feedback="The package arrived quickly and in great shape!",
)

# Scenario 2: Complaint
print("\nRunning Scenario 2: Complaint")
coordinator.run_lifecycle(
    customer_input={"items": ["Monitor"], "total": 199.99},
    feedback="The box was damaged and the screen is broken. Terrible service.",
)
```